# Calibration on the bundled sample data — outputs per instrument

Runs both calibrations on the small real fixtures under `examples/data/` (see
`examples/data/README.md`) and **displays each instrument's output inline**:

- **Rayleigh** (CHM15k 1064 nm, Mini-MPL 532 nm, CL61 910 nm) → the diagnostic dashboard.
- **Cloud** (CL31, CL51, CL61 — all 910 nm) → the per-profile calibration coefficients.

CHM15k and Mini-MPL run fully offline; the 910 nm instruments need a monthly **CAMS**
file for the water-vapour correction (the WV LUT is bundled), so those are guarded.

## 1. Setup

In [ ]:
import tempfile
from pathlib import Path
from netCDF4 import Dataset
import numpy as np
from IPython.display import Image, display

from calibration import calibrate_rayleigh, CalibrationOptions, InstrumentInfo
from calibration.config import InstrumentType, DataLevel
from calibration.cloud import liquid_cloud_calibration, CloudCalConfig

# ─── Switch the WHOLE notebook between L1 and L2 here ──────────────────────────
LEVEL = "L2"          # "L1" = raw range-corrected signal | "L2" = attenuated backscatter
# ──────────────────────────────────────────────────────────────────────────────

ROOT = Path.cwd()
if not (ROOT / "examples" / "data").exists() and (ROOT.parent / "examples" / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"
OPTIONS = ROOT / "options.json"
CAMS = Path(CalibrationOptions.from_json(OPTIONS).cams_folder)
DLEVEL = {"L1": DataLevel.L1, "L2": DataLevel.L2_DAILY}[LEVEL]
print(f"sample data : {DATA}")
print(f"data level  : {LEVEL}  ({DLEVEL.name})")
print(f"CAMS present: {CAMS.exists()} (needed only for the 910 nm instruments)")

def data_file(wmo, date):
    """Bundled file for the CURRENT level — flip LEVEL above to switch L1<->L2."""
    return DATA / LEVEL / wmo / "2026" / date[4:6] / f"{LEVEL}_{wmo}_A{date}.nc"

def info_from(wmo, itype, ncpath):
    with Dataset(ncpath) as ds:
        lat = float(ds.variables["station_latitude"][...])
        lon = float(ds.variables["station_longitude"][...])
        alt = float(ds.variables["station_altitude"][...])
    return InstrumentInfo(site_name=wmo, wmo_id=wmo, identifier="A",
                          instrument_type=itype, latitude=lat, longitude=lon, altitude=alt)

def rayleigh_options(plots=False):
    o = CalibrationOptions.from_json(OPTIONS)
    o.data_level = DLEVEL
    o.folder_root = DATA / LEVEL
    o.cams_folder = CAMS
    o.abs_cs_lookup_table = Path("")          # use the bundled WV LUT
    o.folder_output = Path(tempfile.mkdtemp())
    o.plot_main = bool(plots)                 # -> ONLY the compact diagnostics dashboard
    o.plot_all = False                        # no simple per-step RCS plots
    return o

## 2. Rayleigh outputs (per instrument)

For each molecular instrument: run the calibration with plotting on, print the result,
and display the compact diagnostic dashboard. The reported constant is the **Wiegner lidar
constant** `C_L = RCS/β_att`.

> **L1 vs L2.** On **L2** these fixtures calibrate cleanly. On **L1** the molecular fit often
> flags `-2` ("signal not proportional to molecular"): the raw `rcs_0` retains a high-altitude
> background / afterpulse residual (it *grows* like r² aloft) that the L1→L2 processing removes
> from `attenuated_backscatter_0`. The molecular fit needs that cleaned signal, so **L2 is the
> working default**; flip `LEVEL="L1"` to see the difference on identical days.

In [ ]:
RAYLEIGH = [
    ("CHM15k",   "0-20000-0-06610", "20260225", InstrumentType.CHM15k),
    ("Mini-MPL", "0-20000-0-07014", "20260423", InstrumentType.MINI_MPL),
    ("CL61",     "0-756-4-EERLCL61", "20260304", InstrumentType.CL61),
]
for typ, wmo, date, itype in RAYLEIGH:
    f = data_file(wmo, date)
    if not f.exists():
        print(f"{typ}: no {LEVEL} fixture for {date} - skipped"); continue
    if itype is InstrumentType.CL61 and not CAMS.exists():
        print(f"{typ}: needs CAMS for the 910 nm WV correction - skipped"); continue
    o = rayleigh_options(plots=True)
    res = calibrate_rayleigh(date, info_from(wmo, itype, f), o)
    note = "  (L1 rcs_0 background residual - see note above)" if (LEVEL == "L1" and res.flag < 0) else ""
    print(f"=== {typ} ({wmo})  {date}  [{LEVEL}]  —  Rayleigh: flag={res.flag} "
          f"({res.flag_meaning}), C_L={res.lidar_constant:.3e} ==={note}")
    pdir = o.folder_output / "plots" / wmo / date[:4]
    compact = pdir / f"{date}_{wmo}_rayleigh_diag_compact.png"
    if compact.exists():
        display(Image(filename=str(compact)))        # ONLY the compact diagnostic dashboard
    else:
        print("   (calibration flagged before the fit completed - no compact dashboard)")

## 3. Cloud outputs (per instrument)

For each cloud instrument: run the liquid-cloud (O'Connor/Hopkin) calibration, print the
result, and plot the per-profile **lidar constant `C_L`**.

Following the package-wide Wiegner convention, the headline is `C_L = RCS/β_att` (the *same*
quantity as Rayleigh). The cloud method's native output is the O'Connor multiplier
`C = β_true/β_file (≈1)`; it is converted via **`C_L = calibration_constant_0 / C`**. On **L2**
the file carries `calibration_constant_0`, so an absolute `C_L` is shown; on **L1** there is no
applied constant, so the dimensionless Wiegner-sense inverse **`1/C`** is shown instead (and
labelled as such). See `doc/reports/calibration_coefficient_convention.md`. The liquid-cloud
method is suitable for every elastic ceilometer/lidar and is the **primary** method for CL31/CL51.

In [ ]:
import matplotlib
matplotlib.use("Agg")            # save to PNG, then display (backend-agnostic)
import matplotlib.pyplot as plt

# Per-level dates: some streams have different bundled days at L1 vs L2.
CLOUD = [
    ("CL31", "0-20000-0-06602",  {"L1": "20260220", "L2": "20260220"}),
    ("CL51", "0-20000-0-02998",  {"L1": "20260409", "L2": "20260116"}),
    ("CL61", "0-756-4-EERLCL61", {"L1": "20260304", "L2": "20260304"}),
]
if not CAMS.exists():
    print("CAMS not found - cloud outputs skipped (910 nm WV correction).")
else:
    for typ, wmo, dates in CLOUD:
        date = dates[LEVEL]
        f = data_file(wmo, date)
        if not f.exists():
            print(f"{typ}: no {LEVEL} fixture for {date} - skipped"); continue
        r = liquid_cloud_calibration(CloudCalConfig(
            nc_file=str(f), cams_folder=str(CAMS),
            apply_wv_correction=1, aerosol_lidar_ratio=50))      # bundled WV LUT
        C = np.asarray(r.all_coefficients, float).ravel()        # per-profile O'Connor multiplier
        # Wiegner convention: report C_L = calibration_constant_0 / C. On L1 there is no applied
        # constant (lidar_constant is NaN) -> show the dimensionless inverse 1/C instead.
        if np.isfinite(r.lidar_constant):
            series = r.calibration_constant_applied / C
            med, ylab = r.lidar_constant, r"lidar constant $C_L$ (= calConst / C)"
            hdr = f"C_L={r.lidar_constant:.3e}"
        else:
            series = 1.0 / C
            med, ylab = r.calibration_factor, r"Wiegner-sense inverse $1/C$ (no calConst at L1)"
            hdr = f"1/C={r.calibration_factor:.3f}  (no absolute C_L at L1)"
        print(f"=== {typ} ({wmo})  {date}  [{LEVEL}]  —  cloud: in-cloud profiles "
              f"n={r.n_profiles}, {hdr} ===")
        fig, ax = plt.subplots(figsize=(11, 3.2))
        ax.plot(series, ".", ms=5, alpha=0.7, color="#1f77b4", label="per-profile")
        if np.isfinite(med):
            ax.axhline(med, color="red", lw=1.3, label=f"median = {med:.3e}")
        ax.set_xlabel("profile index (in-cloud profiles are the non-NaN points)")
        ax.set_ylabel(ylab)
        ax.set_title(f"{typ} ({wmo}) {date} [{LEVEL}] — liquid-cloud calibration (n={r.n_profiles})")
        ax.legend(fontsize=8); ax.grid(alpha=0.3); fig.tight_layout()
        png = Path(tempfile.mkdtemp()) / f"{typ}_{date}_cloud.png"
        fig.savefig(png, dpi=120); plt.close(fig)
        display(Image(filename=str(png)))

Flip `LEVEL` in the setup cell to recalibrate the same instruments on **L1** (raw
range-corrected signal) instead of **L2** (attenuated backscatter) — useful for seeing
why the molecular fit behaves differently between levels.

The Rayleigh dashboard's large bottom-left panel is the **full range-corrected-signal
matrix**: the gold band is the molecular fit window, white dots are detected cloud bases,
and **hatched columns mark excluded profiles** (red `///` = low cloud, grey `\\\` =
screened / not used). Only the compact dashboard is shown (set `plot_all=True` for the
extra per-step RCS plots). See `tests/test_sample_data.py` for these fixtures as
automated tests and `examples/data/README.md` for the full list.